### Imports

In [1]:
# the "> /dev/null 2>&1" at the end of each line simply suppresses the 
# installation log
!pip3 install rocketcea > /dev/null 2>&1 
# in case it's not installed already (will be used to model NASA CEA)
!pip3 install rocketisp > /dev/null 2>&1 
# used to model chamber and nozzle losses
!pip3 install ambiance > /dev/null 2>&1 
# this is the standard atmosphere model that we will use 
# later in the notebook
!pip3 install cantera > /dev/null 2>&1 
# thermophysical, fluids, and kinetics toolbox
!pip3 install ipywidgets > /dev/null 2>&1 
# for the engien design widget at the end

In [2]:
from rocketcea.cea_obj_w_units import CEA_Obj

In [3]:
from rocketcea.cea_obj_w_units import CEA_Obj
import numpy as np
import matplotlib.pyplot as plt
from ambiance import Atmosphere
import multiprocessing
from rocketisp.rocket_isp import RocketThruster
from rocketisp.geometry import Geometry
from rocketisp.stream_tubes import CoreStream
from rocketisp.efficiencies import Efficiencies
from rocketisp.nozzle.nozzle import Nozzle
from rocketisp.geometry import solidCylVol, solidFrustrumVol
from scipy.optimize import minimize
from scipy.optimize import fsolve
from scipy.interpolate import griddata
import ipywidgets as widgets
from IPython.display import display
import cantera as ct
from PIL import Image
import csv

In [5]:
rocket = CEA_Obj(oxName='LOX', fuelName='RP-1', temperature_units='degK', 
                 cstar_units='m/sec', specific_heat_units='kJ/kg degK', 
                 sonic_velocity_units='m/s', enthalpy_units='J/kg', 
                 density_units='kg/m^3')
# This rocket object will be used going forward
# This initialization assumes an inifite area combustor. 
# This is will later be changed

In [38]:
# pip install CoolProp numpy
from CoolProp.CoolProp import PropsSI
import numpy as np

### Equivalence Ratio

In [7]:
def ER(fMM, oMM, fCoefA, oCoef, MR):
    MRStoich = (oCoef * oMM) / (fCoefA * fMM)
    return MRStoich / MR

### Reactants

In [6]:
fuel = 'C12H26'
oxidizer = 'O2'
fMM = 170.32
oMM = 32
fCoefA = 1
oCoef = 18.5
MR = 2

In [8]:
ER = ER(fMM, oMM, fCoefA, oCoef, 2)
print(ER)

1.737905119774542


In [9]:
fCoefS = ER
C = 12
H = 26
O = 2
nC = fCoefS * C
nH = fCoefS * H
nO = oCoef * O

### Products

#### CO, H2, OH

In [ ]:
#Reactant moles
n1 = 0
n2 = 0
n3 = 0

In [27]:
#        0      1      2
# X = [X_CO,   X_H2,  X_OH]
def linear_equations_X_1(X):
    eq1 = np.sum(X) - 1
    eq2 = X[0] - 2*(nC/nH)*X[1] - (nC/nH)*X[2]
    eq3 = -(nH/nO)*X[0] + 2*X[1] + (1-(nH/nO))*X[2]
    return [eq1, eq2, eq3]

def linear_equations_n_1(X):
    eq1 = X[0] - nC
    eq2 = 2*X[1] + X[2] - nH
    eq3 = X[0] + X[2] - nO
    return [eq1, eq2, eq3]

X0 = [1, 1, 1]
X = fsolve(linear_equations_X_1, X0)
print(f"X | CO: {X[0]:.4f}, H2: {X[1]:.4f}, OH: {X[2]:.4f}")

n0 = [1, 1, 1]
[n1, n2, n3] = fsolve(linear_equations_n_1, n0)
print(f"n | CO: {n1:.4f} mol, H2: {n2:.4f} mol, OH: {n3:.4f} mol")


X | CO: 0.4048, H2: 0.2818, OH: 0.3134
n | CO: 20.8549 mol, H2: 14.5202 mol, OH: 16.1451 mol


#### CO2, H2O, CO

In [31]:
#        0      1      2
# X = [X_CO2, X_H2O,  X_CO]
# def linear_equations_X_2(X):
#     eq1 = np.sum(X) - 1
#     eq2 = X[0] - 2*(nC/nH)*X[1] - (nC/nH)*X[2]
#     eq3 = -(nH/nO)*X[0] + 2*X[1] + (1-(nH/nO))*X[2]
#     return [eq1, eq2, eq3]

def linear_equations_n_2(X):
    eq1 = X[0] + X[2] - nC
    eq2 = 2*X[1] - nH
    eq3 = 2*X[0] + X[1] + X[2] - nO
    return [eq1, eq2, eq3]

# X0 = [1, 1, 1]
# X = fsolve(linear_equations_X_2, X0)
# print(f"CO2: {X[0]:.4f}, H2O: {X[1]:.4f}, CO: {X[2]:.4f}")

n0 = [1, 1, 1]
[n1, n2, n3] = fsolve(linear_equations_n_2, n0)
print(f"n | CO2: {n1:.4f} mol, H2o: {n2:.4f} mol, CO: {n3:.4f} mol")

# DOES NOT BALANCE


n | CO2: -6.4476 mol, H2o: 22.5928 mol, CO: 27.3025 mol


#### H2O, CO, H2

In [33]:
#        0       1       2
# X = [X_H2O,   X_CO,  X_H2]
# def linear_equations_X_3(X):
#     eq1 = np.sum(X) - 1
#     eq2 = X[0] - 2*(nC/nH)*X[1] - (nC/nH)*X[2]
#     eq3 = -(nH/nO)*X[0] + 2*X[1] + (1-(nH/nO))*X[2]
#     return [eq1, eq2, eq3]

def linear_equations_n_3(X):
    eq1 = X[1] - nC
    eq2 = 2*X[0] + 2*X[2] - nH
    eq3 = X[0] + X[1] - nO
    return [eq1, eq2, eq3]

# X0 = [1, 1, 1]
# X = fsolve(linear_equations_X_3, X0)
# print(f"X | H2O: {X[0]:.4f}, CO: {X[1]:.4f}, H2: {X[2]:.4f}")

n0 = [1, 1, 1]
[n1, n2, n3] = fsolve(linear_equations_n_3, n0)
print(f"n | H2O: {n1:.4f} mol, CO: {n2:.4f} mol, H2: {n3:.4f} mol")


n | H2O: 16.1451 mol, CO: 20.8549 mol, H2: 6.4476 mol


In [34]:
#Check moles
print(n1)
print(n2)
print(n3)

16.14513856270549
20.854861437294506
6.447627994363557


In [62]:
n1MM = 0.018
n2MM = 0.028
n3MM = 0.002

### Enthalpy

#### Enthalpy Reactants

In [58]:
#Enthalpy units: kJ/kg
Tref = 298.15

fEForm = -1.45e2
fT = 298.15
fESens = (PropsSI('H', 'T', fT, 'P', 1.0e5, 'n-Dodecane') - PropsSI('H', 'T', Tref, 'P', 1.0e5, 'n-Dodecane')) / 1000

oEForm = 0
oT = 90.17
oESens = (PropsSI('H', 'T', oT, 'P', 1.0e5, 'Oxygen') - PropsSI('H', 'T', Tref, 'P', 1.0e5, 'Oxygen')) / 1000


print(fESens)
print(oESens)

0.0
-191.31206460637947


In [61]:
fH = fEForm + fESens
oH = oEForm + oESens

print(fH)
print(oH)

Ein = (fCoefS * fMM / 1000 * fH) + (oCoef * oMM / 1000 * oH)
print(Ein)

-145.0
-191.31206460637947
-156.17674224697663


#### Enthalpy Products

In [69]:
n1EForm = -1.34e4
n2EForm = -3.95e3
n3EForm = 0

n1M = n1 * n1MM
n2M = n2 * n2MM
n3M = n3 * n3MM

n1Name = 'Water'
n2Name = 'CarbonMonoxide'
n3Name = 'Hydrogen'

In [70]:
EFormOut = (n1M * n1EForm) + (n2M * n2EForm) + (n3M * n3EForm)

TARGET_KJ = Ein - EFormOut

print(TARGET_KJ)

6044.57835404236


### Temperature

In [71]:
# ---------- sensible enthalpy relative to 298.15 K using CoolProp ----------
def h_sensible_kJkg_coolprop(fluid: str, T: float, p_Pa: float = 1.0e5) -> float:
    """Return sensible enthalpy h(T)-h(298.15 K) in kJ/kg for a CoolProp fluid."""
    hT   = PropsSI('H', 'T', T,        'P', p_Pa, fluid)  # J/kg
    href = PropsSI('H', 'T', 298.15,   'P', p_Pa, fluid)  # J/kg
    return (hT - href) / 1000.0  # kJ/kg

# ---------- residual function for the energy balance ----------
def residual(T: float) -> float:
    h_n1  = h_sensible_kJkg_coolprop(n1Name, T)  # kJ/kg
    h_n2  = h_sensible_kJkg_coolprop(n2Name, T)        # kJ/kg
    h_n3  = h_sensible_kJkg_coolprop(n3Name, T)                          # kJ/kg
    return (n1M*h_n1 + n2M*h_n2 + n3M*h_n3) - TARGET_KJ

# ---------- bisection solver ----------
def solve_T_bisection(T_lo: float = 1000.0, T_hi: float = 5000.0, tolT: float = 1e-3, maxit: int = 200):
    f_lo = residual(T_lo)
    f_hi = residual(T_hi)
    if f_lo * f_hi > 0:
        raise RuntimeError(f"Bracket does not change sign: F({T_lo})={f_lo:.3f}, F({T_hi})={f_hi:.3f}")
    a, b, fa, fb = T_lo, T_hi, f_lo, f_hi
    for _ in range(maxit):
        c = 0.5*(a + b)
        fc = residual(c)
        if abs(fc) < 1e-3 or (b - a) < tolT:
            return c, fc
        if fa * fc <= 0:
            b, fb = c, fc
        else:
            a, fa = c, fc
    return c, fc

if __name__ == "__main__":
    T_f, res = solve_T_bisection()
    print(f"Adiabatic flame temperature ≈ {T_f:.1f} K  (residual {res:.3f} kJ)")


Adiabatic flame temperature ≈ 3404.6 K  (residual -0.001 kJ)


##### OH Solver (Not in CoolProp)

In [ ]:
MW_OH = 0.017007  # kg/mol
T_pts = np.array([298.15, 300, 500, 800, 1000, 1200, 1600, 2000, 2500, 3000, 3500, 4000], dtype=float)
y_kJ_per_kmol = np.array([0, 0.055, 6.079, 15.467, 21.888, 28.432, 41.838, 55.506, 72.827, 90.315, 107.89, 125.50], dtype=float)

def h_sensible_kJkg_OH(T: float) -> float:
    """Sensible enthalpy for OH in kJ/kg using your tabulated (h_T - h_298) [kJ/kmol]."""
    # linear interpolation over your table
    h_kJ_per_kmol = np.interp(T, T_pts, y_kJ_per_kmol)
    return h_kJ_per_kmol / MW_OH  # → kJ/kg

In [14]:
h_CO  = h_sensible_kJkg_coolprop('Oxygen', 90.17)
print(h_CO)

-191.31206460637947


|  | CEA | CO, H2, OH | H2O, CO, H2 |
|:--------:|:--------:|:--------:|:--------:|
|  Adiabatic Flame Temperature (K)   |  ~   |  1253.7 K   | 3404.6 K   |
|  Chamber Molecular Weight   |  ~   |  ~   |  ~   |
|  Chamber Specific Heat Ratio   |  ~   |  ~   |  ~   |
|  Exhaust Mach Number   |  ~   |  ~   |  ~   |
|  Exhaust Temperature (K)   |  ~   |  ~   |  ~   |
|  Exhaust Velocity (m/s)   |  ~   |  ~   |  ~   |
|  Specific Impulse (s)   |  ~   |  ~   |  ~   |